# Notebook 3: Building a RAG Chain with Conversation History

**Learning Goal**: Understand LangChain Expression Language (LCEL), build a simple RAG chain, then extend it with conversation history awareness using `RunnablePassthrough`, `RunnableBranch`, and the complete chain from `rag_engine.py`.

**rag_engine.py coverage**: This notebook covers `get_rag_chain()` (lines 48-118) — the core chain that powers the FAQ chatbot.

**Prerequisites**: Complete Notebooks 1 and 2 first.

---

## Setup

We need to load the FAISS index created in the previous notebook (or use the production index).

In [ ]:
# %pip install langchain langchain-openai langchain-community faiss-cpu pypdf python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableBranch
from langchain_core.messages import AIMessage, HumanMessage

# Load the production FAISS index
FAISS_INDEX_PATH = os.path.join("..", "faiss_index")

embeddings = OpenAIEmbeddings()
vectorstore = FAISS.load_local(FAISS_INDEX_PATH, embeddings, allow_dangerous_deserialization=True)
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})  # same as rag_engine.py line 57

llm = ChatOpenAI(model="gpt-4o", temperature=0)  # same as rag_engine.py line 59

print("Setup complete: retriever and LLM ready")

## 1. LCEL Basics (LangChain Expression Language)

LCEL is LangChain's way of composing chains using the **pipe operator** (`|`). Each component in a chain is a **Runnable** — an object with an `.invoke()` method that takes input and produces output.

### The Pipe Operator

```python
chain = component_a | component_b | component_c
```

This creates a chain where:
1. Input goes into `component_a`
2. Output of `a` goes into `component_b`
3. Output of `b` goes into `component_c`
4. Output of `c` is the final result

### RunnablePassthrough

`RunnablePassthrough` passes its input through unchanged. Its real power is `.assign()`, which **adds new keys** to a dictionary while keeping existing ones.

```python
RunnablePassthrough.assign(new_key=some_function)
```

This takes an input dict like `{"a": 1}`, runs `some_function` on it, and returns `{"a": 1, "new_key": result}`.

> **In `rag_engine.py`** (lines 100, 109): `RunnablePassthrough.assign()` is used to build up the data dictionary step by step.

In [ ]:
# RunnablePassthrough passes input through unchanged
passthrough = RunnablePassthrough()
result = passthrough.invoke("hello")
print(f"Passthrough: 'hello' -> '{result}'")

# .assign() adds new keys to a dict
chain = RunnablePassthrough.assign(
    uppercased=lambda x: x["text"].upper()
)

result = chain.invoke({"text": "hello world"})
print(f"\nAssign: {{text: 'hello world'}} -> {result}")
# Output: {"text": "hello world", "uppercased": "HELLO WORLD"}

### RunnableBranch

`RunnableBranch` implements **conditional logic** — it runs different chains based on conditions, similar to if/else.

```python
RunnableBranch(
    (condition_func, chain_if_true),   # if condition_func(input) is True
    default_chain                       # else
)
```

> **In `rag_engine.py`** (lines 101-107): `RunnableBranch` checks if chat history exists. If yes, it rewrites the question. If no, it passes the question through unchanged.

In [ ]:
# RunnableBranch: conditional logic
branch = RunnableBranch(
    # Condition: if the input has items
    (lambda x: len(x["items"]) > 0, lambda x: f"Processing {len(x['items'])} items"),
    # Default: no items
    lambda x: "No items to process"
)

print(branch.invoke({"items": [1, 2, 3]}))  # "Processing 3 items"
print(branch.invoke({"items": []}))          # "No items to process"

## 2. Simple RAG Chain (No History)

Before adding history awareness, let's build a basic RAG chain. The flow is:

```
User Question → Retriever → format_docs → Prompt (with context) → LLM → String
```

In [ ]:
# Helper function — same as rag_engine.py lines 45-46
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Simple QA prompt (no history)
simple_qa_prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "You are a helpful assistant. Use the following context to answer the question. "
     "If you don't know the answer, say that you don't know. "
     "Use three sentences maximum.\n\nContext:\n{context}"),
    ("human", "{input}"),
])

# Simple RAG chain: retrieve → format → prompt → llm → parse
simple_rag_chain = (
    RunnablePassthrough.assign(
        context=lambda x: format_docs(retriever.invoke(x["input"]))
    )
    | simple_qa_prompt
    | llm
    | StrOutputParser()
)

# Test it
result = simple_rag_chain.invoke({"input": "What are the NHIF benefit packages?"})
print(result)

This works for single questions. But what happens with follow-ups?

## 3. The Problem with Follow-up Questions

Consider this conversation:
1. User: *"What is NHIF?"*
2. Bot: *"NHIF is Tanzania's National Health Insurance Fund..."*
3. User: *"How much does it cost?"*

The third question uses **"it"** to refer to NHIF from the history. But the retriever doesn't know about the conversation — it only sees *"How much does it cost?"* and can't find relevant documents because *"it"* is ambiguous.

### Solution: Query Contextualization

Before searching, we use the LLM to **rewrite** the follow-up question as a **standalone question**:

```
"How much does it cost?" → "How much does NHIF cost?"
```

Now the retriever can find relevant documents!

In [ ]:
# Demonstrate the problem: search with an ambiguous follow-up
ambiguous_query = "How much does it cost?"
results = retriever.invoke(ambiguous_query)

print(f"Query: '{ambiguous_query}'")
print(f"\nRetrieved {len(results)} documents. First result:")
print(results[0].page_content[:200] + "...")
print("\n(These results may not be about NHIF costs because 'it' is ambiguous)")

In [ ]:
# Now search with the contextualized version
standalone_query = "How much does NHIF health insurance cost?"
results = retriever.invoke(standalone_query)

print(f"Query: '{standalone_query}'")
print(f"\nRetrieved {len(results)} documents. First result:")
print(results[0].page_content[:200] + "...")
print("\n(Much more relevant results!)")

## 4. Building the Contextualization Chain

This is the chain from `rag_engine.py` (lines 62-78) that rewrites follow-up questions into standalone questions.

### Production Code Reference
```python
# rag_engine.py lines 62-78
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

cntx_q_chain = contextualize_q_prompt | llm | StrOutputParser()
```

In [ ]:
# Build the contextualization chain — exact same as rag_engine.py

contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

cntx_q_chain = contextualize_q_prompt | llm | StrOutputParser()

print("Contextualization chain created")

In [ ]:
# Test: rewrite a follow-up question
chat_history = [
    HumanMessage(content="What is NHIF?"),
    AIMessage(content="NHIF is Tanzania's National Health Insurance Fund."),
]

standalone = cntx_q_chain.invoke({
    "chat_history": chat_history,
    "input": "How much does it cost?"
})

print(f"Follow-up:   'How much does it cost?'")
print(f"Standalone:  '{standalone}'")

In [ ]:
# Test: a question that doesn't need rewriting
standalone = cntx_q_chain.invoke({
    "chat_history": [],
    "input": "What are the NHIF benefit packages?"
})

print(f"Original:    'What are the NHIF benefit packages?'")
print(f"Standalone:  '{standalone}'")
print("(Should be returned as-is since there's no history)")

## 5. Conditional Logic with RunnableBranch

We don't need to run the contextualization chain on the first message (when there's no history). `RunnableBranch` lets us **skip** it when chat history is empty.

### Production Code Reference
```python
# rag_engine.py lines 101-107
RunnableBranch(
    (
        lambda x: len(x.get("chat_history", [])) > 0,  # condition
        cntx_q_chain                                     # if True: rewrite
    ),
    lambda x: x["input"]                                 # else: return as-is
)
```

This reads as:
- **If** chat history has items → run the contextualization chain
- **Else** → just return the raw input question

In [ ]:
# Build the conditional question handler
question_handler = RunnableBranch(
    (
        lambda x: len(x.get("chat_history", [])) > 0,
        cntx_q_chain
    ),
    lambda x: x["input"]
)

# Test with history (should rewrite)
result_with_history = question_handler.invoke({
    "chat_history": [
        HumanMessage(content="What is NHIF?"),
        AIMessage(content="NHIF is Tanzania's National Health Insurance Fund."),
    ],
    "input": "How much does it cost?"
})
print(f"With history:    '{result_with_history}'")

# Test without history (should pass through)
result_no_history = question_handler.invoke({
    "chat_history": [],
    "input": "What are the NHIF benefit packages?"
})
print(f"Without history: '{result_no_history}'")

## 6. The Complete History-Aware RAG Chain

Now let's put everything together. This is the full `get_rag_chain()` from `rag_engine.py` (lines 99-115).

### Data Flow

```
Input: {"input": "...", "chat_history": [...]}
  │
  ├─ Step 1: RunnablePassthrough.assign(standalone_question=...)
  │    └─ RunnableBranch:
  │         ├─ Has history? → cntx_q_chain (rewrite question)
  │         └─ No history?  → return input as-is
  │    Result: {"input": ..., "chat_history": ..., "standalone_question": "..."}
  │
  ├─ Step 2: RunnablePassthrough.assign(context=...)
  │    └─ retriever.invoke(standalone_question) → format_docs
  │    Result: {"input": ..., "chat_history": ..., "standalone_question": ..., "context": "..."}
  │
  ├─ Step 3: qa_prompt
  │    └─ Format system message with {context}, inject {chat_history}, add {input}
  │
  ├─ Step 4: llm
  │    └─ Generate response based on context and history
  │
  └─ Step 5: StrOutputParser
       └─ Extract string from AIMessage
       
Output: "The answer is..."
```

### Production Code Reference
```python
# rag_engine.py lines 80-115
qa_system_prompt = (
    "You are a helpful assistant for answering frequently asked questions. "
    "Use the following pieces of retrieved context to answer the question. "
    "If the user asks in English, answer in English. "
    "If the user asks in Kiswahili, answer in Kiswahili. "
    "If you don't know the answer, say that you don't know. "
    "Use three sentences maximum and keep the answer concise."
    "\n\n"
    "Context:\n{context}"
)

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

history_aware_retriever = (
    RunnablePassthrough.assign(
        standalone_question=RunnableBranch(
            (lambda x: len(x.get("chat_history", [])) > 0, cntx_q_chain),
            lambda x: x["input"]
        )
    )
    | RunnablePassthrough.assign(
        context=lambda x: format_docs(retriever.invoke(x["standalone_question"]))
    )
    | qa_prompt
    | llm
    | StrOutputParser()
)
```

In [ ]:
# Build the complete chain — exactly as in rag_engine.py

# QA prompt with bilingual support (lines 80-97)
qa_system_prompt = (
    "You are a helpful assistant for answering frequently asked questions. "
    "Use the following pieces of retrieved context to answer the question. "
    "If the user asks in English, answer in English. "
    "If the user asks in Kiswahili, answer in Kiswahili. "
    "If you don't know the answer, say that you don't know. "
    "Use three sentences maximum and keep the answer concise."
    "\n\n"
    "Context:\n{context}"
)

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# The complete history-aware RAG chain (lines 99-115)
history_aware_retriever = (
    # Step 1: Get standalone question (rewrite if history exists)
    RunnablePassthrough.assign(
        standalone_question=RunnableBranch(
            (
                lambda x: len(x.get("chat_history", [])) > 0,
                cntx_q_chain
            ),
            lambda x: x["input"]
        )
    )
    # Step 2: Retrieve and format context
    | RunnablePassthrough.assign(
        context=lambda x: format_docs(retriever.invoke(x["standalone_question"]))
    )
    # Step 3-5: Format prompt, send to LLM, parse output
    | qa_prompt
    | llm
    | StrOutputParser()
)

print("History-aware RAG chain created!")

## 7. Testing the Chain

Let's test the chain with both single questions and multi-turn conversations.

In [ ]:
# Test 1: First question (no history)
response = history_aware_retriever.invoke({
    "input": "What is NHIF?",
    "chat_history": []
})

print("Q: What is NHIF?")
print(f"A: {response}")

In [ ]:
# Test 2: Follow-up question (with history)
chat_history = [
    HumanMessage(content="What is NHIF?"),
    AIMessage(content=response),  # use the response from above
]

response2 = history_aware_retriever.invoke({
    "input": "What benefits does it provide?",  # "it" refers to NHIF
    "chat_history": chat_history
})

print("Q: What benefits does it provide?")
print(f"A: {response2}")

In [ ]:
# Test 3: Multi-turn conversation simulation
print("=" * 60)
print("MULTI-TURN CONVERSATION")
print("=" * 60)

questions = [
    "What are the NHIF benefit packages?",
    "Which one is for hospitals?",
    "How much does it cost?",
]

chat_history = []

for question in questions:
    # Invoke the chain
    answer = history_aware_retriever.invoke({
        "input": question,
        "chat_history": chat_history
    })
    
    print(f"\nUser: {question}")
    print(f"Bot:  {answer}")
    
    # Update chat history for next turn
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=answer))

In [ ]:
# Test 4: Kiswahili support
response_sw = history_aware_retriever.invoke({
    "input": "NHIF ni nini?",
    "chat_history": []
})

print("Q: NHIF ni nini?")
print(f"A: {response_sw}")

## How This Connects to the Production App

The chain we just built is exactly what powers the FAQ chatbot. Here's how it's used in `main.py` (the Streamlit app):

```python
# main.py lines 53-62
rag_chain = get_rag_chain()  # returns our history_aware_retriever

response_text = rag_chain.invoke({
    "input": prompt,           # user's question
    "chat_history": chat_history  # list of HumanMessage/AIMessage
})
```

The Streamlit app manages:
- **Session state**: Stores messages across page reloads
- **Chat history conversion**: Converts stored messages to `HumanMessage`/`AIMessage` objects
- **UI**: Chat interface with message bubbles

The Telegram bot (`telegram_bot.py`) uses the same chain in a similar way.

## Summary

In this notebook we learned:

| Concept | What we learned | rag_engine.py reference |
|---|---|---|
| LCEL (`\|` pipe) | Chain components together left-to-right | Lines 78, 99-114 |
| `RunnablePassthrough.assign()` | Add computed keys to a dictionary | Lines 100, 109 |
| `RunnableBranch` | Conditional logic (if/else) in chains | Lines 101-107 |
| Query contextualization | Rewrite follow-ups as standalone questions | Lines 62-78 |
| `format_docs()` | Convert documents to a context string | Lines 45-46 |
| History-aware RAG | Full chain: contextualize → retrieve → prompt → answer | Lines 99-115 |

### Complete Architecture

```
Notebook 1 (Fundamentals)     Notebook 2 (Documents)      Notebook 3 (RAG Chain)
──────────────────────       ────────────────────         ─────────────────────
ChatOpenAI                   PyPDFLoader                  RunnablePassthrough
HumanMessage / AIMessage     RecursiveCharacterTextSplitter   RunnableBranch
ChatPromptTemplate           OpenAIEmbeddings             Contextualization chain
MessagesPlaceholder          FAISS                        History-aware RAG chain
StrOutputParser              Retriever                    Complete get_rag_chain()
Pipe operator (|)            format_docs()
```

You now understand every component of `rag_engine.py`!